In [ ]:
# ---------------------------------
# 0. Setup & Imports
# ---------------------------------
import os
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv
import gradio as gr

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# ---------------------------------
# 1. Load and Embed Documents
# ---------------------------------
manual = TextLoader(
    "C:/Users/admin/Desktop/New_GenAI/GenAI/LangGraph/Autonomus RAG/xumo_manual_rag.txt",
    encoding="utf-8"
).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(manual)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

# ---------------------------------
# 2. Initialize Groq LLM
# ---------------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

# ---------------------------------
# 3. Define Agent State
# ---------------------------------
class IterativeReflectRAGState(BaseModel):
    question: str
    refined_question: str = ""
    retrieved_docs: List[Document] = []
    answer: str = ""
    verified: bool = False
    attempts: int = 0

# ---------------------------------
# 4. Nodes
# ---------------------------------

# a. Retrieve
def retrieve_docs(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    query = state.refined_question or state.question
    docs = retriever.invoke(query)
    return state.model_copy(update={"retrieved_docs": docs})

# b. Generate Answer
def generate_answer(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)
    prompt = f"""
Use the following context to answer the question:

Context:
{context}

Question:
{state.question}
"""
    response = llm.invoke(prompt.strip()).content.strip()
    return state.model_copy(update={"answer": response, "attempts": state.attempts + 1})

# c. Reflect
def reflect_on_answer(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    prompt = f"""
Evaluate whether the answer below is factually sufficient and complete.

Question: {state.question}
Answer: {state.answer}

Respond 'YES' if it's complete, otherwise 'NO' with feedback.
"""
    feedback = llm.invoke(prompt).content.lower()
    verified = "yes" in feedback
    return state.model_copy(update={"verified": verified})

# d. Refine Query
def refine_query(state: IterativeReflectRAGState) -> IterativeReflectRAGState:
    prompt = f"""
The answer appears incomplete. Suggest a better version of the query that would help retrieve more relevant context.

Original Question: {state.question}
Current Answer: {state.answer}
"""
    new_query = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"refined_question": new_query})

# ---------------------------------
# 5. Build LangGraph
# ---------------------------------
builder = StateGraph(IterativeReflectRAGState)

builder.add_node("retrieve", retrieve_docs)
builder.add_node("answer", generate_answer)
builder.add_node("reflect", reflect_on_answer)
builder.add_node("refine", refine_query)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", "reflect")

builder.add_conditional_edges(
    "reflect",
    lambda s: END if s.verified or s.attempts >= 2 else "refine"
)

builder.add_edge("refine", "retrieve")
builder.add_edge("answer", END)

graph = builder.compile()

# ---------------------------------
# 6. Gradio Interface
# ---------------------------------
def iterative_rag_pipeline(user_query: str):
    init_state = IterativeReflectRAGState(question=user_query)
    result = graph.invoke(init_state)

    return (
        f"Final Answer:\n{result['answer']}\n\n"
        f"Verified: {result['verified']}\n"
        f"Attempts: {result['attempts']}"
    )

demo = gr.Interface(
    fn=iterative_rag_pipeline,
    inputs=gr.Textbox(label="Ask a complex question about the Xumo Manual"),
    outputs=gr.Textbox(label="Iterative + Reflection RAG Output"),
    title="Xumo Manual Iterative + Reflection RAG Assistant"
)

if __name__ == "__main__":
    demo.launch()


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\admin\AppData\Local\Temp\ipykernel_23940\14982137.py:33: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
